In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment ready for data cleaning.")

Environment ready for data cleaning.


In [36]:
DATA_PATH = Path("../Dataset/raw/GUIDE_Test.csv")

assert DATA_PATH.exists(), "Dataset file not found!"

print("Dataset:", DATA_PATH)
print("File size (GB):", round(DATA_PATH.stat().st_size / 1e9, 2))

Dataset: ..\Dataset\raw\GUIDE_Test.csv
File size (GB): 1.09


In [37]:
columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

print("Total columns:", len(columns))

for number, column in enumerate(columns, start=1):
    print(number, column)

Total columns: 46
1 Id
2 OrgId
3 IncidentId
4 AlertId
5 Timestamp
6 DetectorId
7 AlertTitle
8 Category
9 MitreTechniques
10 IncidentGrade
11 ActionGrouped
12 ActionGranular
13 EntityType
14 EvidenceRole
15 DeviceId
16 Sha256
17 IpAddress
18 Url
19 AccountSid
20 AccountUpn
21 AccountObjectId
22 AccountName
23 DeviceName
24 NetworkMessageId
25 EmailClusterId
26 RegistryKey
27 RegistryValueName
28 RegistryValueData
29 ApplicationId
30 ApplicationName
31 OAuthApplicationId
32 ThreatFamily
33 FileName
34 FolderPath
35 ResourceIdName
36 ResourceType
37 Roles
38 OSFamily
39 OSVersion
40 AntispamDirection
41 SuspicionLevel
42 LastVerdict
43 CountryCode
44 State
45 City
46 Usage


In [38]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=20000,
    low_memory=False
)

print("Sample shape:", sample_df.shape)

sample_df.head()

Sample shape: (20000, 46)


,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City,Usage
0,1245540519230,657,11767,87199,2024-06-04T22:56:27.000Z,524,563,LateralMovement,T1021;T1047;T1105;T1569.002,BenignPositive,NaN,NaN,User,Impacted,98799,138268,360606,160396,2610,3699,425863,863,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Private
1,1400159342154,3,91158,632273,2024-06-03T12:58:26.000Z,2,2,CommandAndControl,NaN,BenignPositive,NaN,NaN,Machine,Impacted,1239,138268,360606,160396,441377,673934,425863,453297,2833,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,0,0,NaN,Suspicious,Suspicious,242,1445,10630,Public
2,1279900255923,145,32247,131719,2024-06-08T03:20:49.000Z,2932,10807,LateralMovement,T1021;T1027.002;T1027.005;T1105,BenignPositive,NaN,NaN,Process,Related,98799,4296,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,14,22,3586,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Public
3,60129547292,222,15294,917686,2024-06-12T12:07:31.000Z,0,0,InitialAccess,T1078;T1078.004,FalsePositive,NaN,NaN,CloudLogonSession,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630,Public
4,515396080539,363,7615,5944,2024-06-06T17:42:05.000Z,27,18,Discovery,T1087;T1087.002,BenignPositive,NaN,NaN,User,Impacted,98799,138268,360606,160396,133549,673934,425863,136104,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,Suspicious,5,66,NaN,NaN,NaN,242,1445,10630,Public


In [39]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Id                  20000 non-null  int64  
 1   OrgId               20000 non-null  int64  
 2   IncidentId          20000 non-null  int64  
 3   AlertId             20000 non-null  int64  
 4   Timestamp           20000 non-null  str    
 5   DetectorId          20000 non-null  int64  
 6   AlertTitle          20000 non-null  int64  
 7   Category            20000 non-null  str    
 8   MitreTechniques     8919 non-null   str    
 9   IncidentGrade       20000 non-null  str    
 10  ActionGrouped       12 non-null     str    
 11  ActionGranular      12 non-null     str    
 12  EntityType          20000 non-null  str    
 13  EvidenceRole        20000 non-null  str    
 14  DeviceId            20000 non-null  int64  
 15  Sha256              20000 non-null  int64  
 16  IpAddress      

In [40]:
missing_counts = pd.Series(
    0,
    index=columns,
    dtype="int64"
)

total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=100000,
    low_memory=False
):
    missing_counts += chunk.isna().sum()
    total_rows += len(chunk)

missing_report = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": (
        missing_counts / total_rows * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    "Missing_Percentage",
    ascending=False
)

print("Total rows processed:", total_rows)

missing_report

Total rows processed: 4147992


,Missing_Count,Missing_Percentage
ActionGrouped,4146079,99.95
ActionGranular,4146079,99.95
ResourceType,4144998,99.93
ThreatFamily,4116614,99.24
EmailClusterId,4106285,98.99
AntispamDirection,4071481,98.16
Roles,4039317,97.38
SuspicionLevel,3498157,84.33
LastVerdict,3155260,76.07
MitreTechniques,2307104,55.62


In [41]:
grade_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["IncidentGrade"],
    chunksize=100000
):
    counts = chunk["IncidentGrade"].value_counts(
        dropna=False
    )

    for label, count in counts.items():
        grade_counts[label] = (
            grade_counts.get(label, 0) + count
        )

grade_report = pd.Series(
    grade_counts,
    name="Record_Count"
).to_frame()

grade_report["Percentage"] = (
    grade_report["Record_Count"] / total_rows * 100
).round(2)

grade_report

,Record_Count,Percentage
BenignPositive,1752940,42.26
TruePositive,1492354,35.98
FalsePositive,902698,21.76


In [42]:
category_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["Category"],
    chunksize=100000
):
    counts = chunk["Category"].value_counts(
        dropna=False
    )

    for label, count in counts.items():
        category_counts[label] = (
            category_counts.get(label, 0) + count
        )

category_report = pd.Series(
    category_counts,
    name="Count"
).sort_values(ascending=False)

category_report.head(20)

InitialAccess          1952969
Exfiltration            615284
SuspiciousActivity      440093
CommandAndControl       357179
Impact                  307810
CredentialAccess        130196
Execution               115658
Malware                  71593
Discovery                63633
Persistence              29075
DefenseEvasion           21611
LateralMovement          19426
UnwantedSoftware          7496
Collection                6272
Ransomware                5605
Exploit                   1653
PrivilegeEscalation       1555
CredentialStealing         873
WebExploit                  11
Name: Count, dtype: int64

In [43]:
timestamp_sample = sample_df["Timestamp"]

parsed_timestamp = pd.to_datetime(
    timestamp_sample,
    errors="coerce",
    utc=True
)

print("Invalid timestamps:",
      parsed_timestamp.isna().sum())

print("Earliest timestamp:",
      parsed_timestamp.min())

print("Latest timestamp:",
      parsed_timestamp.max())

Invalid timestamps: 0
Earliest timestamp: 2024-01-23 20:33:43+00:00
Latest timestamp: 2024-06-17 09:18:16+00:00


In [1]:
from pathlib import Path

current = Path.cwd().resolve()

PROJECT_ROOT = None

for folder in [current, *current.parents]:
    candidate = (
        folder / "Dataset" / "raw" / "GUIDE_Test.csv"
    )

    if candidate.exists():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project dataset."
    )

RAW_PATH = (
    PROJECT_ROOT / "Dataset" / "raw" / "GUIDE_Test.csv"
)

CLEAN_PATH = Path("../Dataset/processed/GUIDE_Test_cleaned_v2.csv")

REPORT_PATH = (
    PROJECT_ROOT / "Dataset" / "processed"
    / "cleaning_report.json"
)

print("Project root:", PROJECT_ROOT)
print("Raw dataset:", RAW_PATH)
print("Cleaned dataset:", CLEAN_PATH)
print("Cleaning report:", REPORT_PATH)

Project root: D:\Microsoft-SOC-Project
Raw dataset: D:\Microsoft-SOC-Project\Dataset\raw\GUIDE_Test.csv
Cleaned dataset: ..\Dataset\processed\GUIDE_Test_cleaned_v2.csv
Cleaning report: D:\Microsoft-SOC-Project\Dataset\processed\cleaning_report.json


In [45]:
raw_rows = 0

for chunk in pd.read_csv(
    RAW_PATH,
    usecols=["Id"],
    dtype={"Id": "string"},
    chunksize=100000,
    low_memory=False
):
    raw_rows += len(chunk)

print("Original dataset records:", raw_rows)

Original dataset records: 4147992


In [46]:
processed_rows = 0

if not CLEAN_PATH.exists():
    print("Cleaned dataset does not exist.")
else:
    for chunk in pd.read_csv(
        CLEAN_PATH,
        usecols=["Id"],
        dtype={"Id": "string"},
        chunksize=100000,
        low_memory=False
    ):
        processed_rows += len(chunk)

    print("Processed records:", processed_rows)

print("Missing records:", raw_rows - processed_rows)

Processed records: 1183092
Missing records: 2964900


In [47]:
import json

if REPORT_PATH.exists():

    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        report = json.load(f)

    print("Rows recorded in cleaning report:",
          report["total_rows"])

    print("Columns:",
          report["total_columns"])

    print("Invalid timestamps:",
          report["invalid_timestamps"])

else:
    print("Cleaning report not found.")

Rows recorded in cleaning report: 4147992
Columns: 46
Invalid timestamps: 0


In [49]:
processed_rows = 0

for chunk in pd.read_csv(
    CLEAN_PATH,
    chunksize=100000,
    usecols=["Id"],
    dtype={"Id": "string"},
    low_memory=False
):
    processed_rows += len(chunk)

print("Processed records:", processed_rows)

Processed records: 1183092


In [50]:
import pandas as pd
import json
from pathlib import Path

CLEAN_PATH = Path("../Dataset/processed/GUIDE_Test_cleaned_v2.csv")
REPORT_PATH = Path("../Dataset/processed/cleaning_report_v2.json")

# 1. Check that both files exist
assert CLEAN_PATH.exists(), "Cleaned CSV not found"
assert REPORT_PATH.exists(), "Cleaning report not found"

# 2. Count actual rows without loading the entire dataset
total_rows = 0

for chunk in pd.read_csv(
    CLEAN_PATH,
    chunksize=100_000,
    usecols=["Id"],
    dtype={"Id": "string"},
    low_memory=False
):
    total_rows += len(chunk)

# 3. Check actual columns
columns = pd.read_csv(CLEAN_PATH, nrows=0).columns.tolist()

# 4. Load audit report
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

# 5. Validate
print("Actual rows:", total_rows)
print("Actual columns:", len(columns))
print("Report rows:", report["total_rows"])
print("Report columns:", report["total_columns"])
print("Invalid timestamps:", report["invalid_timestamps"])

print("\nIncident grades:")
print(report["incident_grades"])

print("\nValidation results:")

assert total_rows == 4_147_992
assert len(columns) == 46
assert report["total_rows"] == total_rows
assert report["total_columns"] == len(columns)
assert sum(report["incident_grades"].values()) == total_rows
assert sum(report["categories"].values()) == total_rows

print("All validation checks passed.")

Actual rows: 4147992
Actual columns: 46
Report rows: 4147992
Report columns: 46
Invalid timestamps: 0

Incident grades:
{'BenignPositive': 1752940, 'TruePositive': 1492354, 'FalsePositive': 902698}

Validation results:
All validation checks passed.
